<a href="https://colab.research.google.com/github/rxnu/LLM-Project/blob/main/sentiment_pretrained_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install --upgrade transformers


In [1]:
!pip install -U "datasets<=2.14.5" "fsspec<=2023.6.0"


In [2]:
!pip install numpy==1.26.0

In [3]:
from datasets import load_dataset

ds = load_dataset('imdb')

ds


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [4]:
ds['train'][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [6]:
ds['train'].features

{'text': Value(dtype='string', id=None),
 'label': ClassLabel(names=['neg', 'pos'], id=None)}

In [5]:
import pandas as pd

ds_train = pd.DataFrame(ds['train'])
ds_test = pd.DataFrame(ds['test'])

ds_train.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [7]:
from datasets import Dataset, DatasetDict
# assign the splits
train = Dataset.from_pandas(ds_train)
test = Dataset.from_pandas(ds_test)
#reconstruct both datasets into a Dataset Dict object
new_ds = DatasetDict({
    'train': train,
    'test': test
})
#view the dataset dict object
new_ds


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})

In [8]:
import pandas as pd
import numpy as np

train_df = pd.DataFrame(ds["train"])
test_df  = pd.DataFrame(ds["test"])

In [9]:
# Q1: How many documents?
n_train = len(train_df)
n_test  = len(test_df)
print(f"Train docs: {n_train:,}, Test docs: {n_test:,}")

Train docs: 25,000, Test docs: 25,000


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# vectorizer with basic cleaning
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=20_000
)

# fit on train reviews, transform both splits
X_train = tfidf.fit_transform(train_df['text'])
X_test  = tfidf.transform(test_df['text'])

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

Vocabulary size: 20000


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# train
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_df['label'])

# predict
y_pred = clf.predict(X_test)
acc = accuracy_score(test_df['label'], y_pred)
print(f"Accuracy: {acc:.4%}\n")
print(classification_report(test_df["label"], y_pred, target_names=["neg","pos"]))

Accuracy: 87.9960%

              precision    recall  f1-score   support

         neg       0.88      0.88      0.88     12500
         pos       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [12]:

# Top Predictive Features (Logistic Regression coefficients)
feature_names = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_[0]
top_pos = feature_names[np.argsort(coefs)[-10:]]
top_neg = feature_names[np.argsort(coefs)[:10]]


print("Top positive features:", top_pos)
print("Top negative features:", top_neg)

Top positive features: ['today' 'fun' 'loved' 'favorite' 'amazing' 'perfect' 'wonderful' 'best'
 'excellent' 'great']
Top negative features: ['worst' 'bad' 'awful' 'waste' 'boring' 'poor' 'worse' 'terrible' 'poorly'
 'dull']


In [13]:
# Inspect Misclassifications
test_texts = test_df["text"].values
false_negatives = np.where((test_df["label"]==1) & (y_pred==0))[0]
false_positives = np.where((test_df["label"]==0) & (y_pred==1))[0]

print("Examples of False Negatives (true=pos, pred=neg):")
for idx in false_negatives[:3]:
    print(f"- {test_texts[idx][:200]!r}...\n")

print("Examples of False Positives (true=neg, pred=pos):")
for idx in false_positives[:3]:
    print(f"- {test_texts[idx][:200]!r}...\n")

Examples of False Negatives (true=pos, pred=neg):
- 'Its a very sensitive portrayal of life with unquenched or constrained desires. What does one do with desire in a culture and society with rigid norms? One husband finds outlet with the immigrant - sin'...

- 'This was a bold movie to hit Indian cinemas when it was released. The first movie to perhaps openly depict lesbian tendencies amongst Indian women. The leading actress of Indian cinema Shabana Azmi ad'...

- 'The theme is controversial and the depiction of the hypocritical and sexually starved india is excellent.Nothing more to this film.There is a lack of good dialogues(why was the movie in english??). Th'...

Examples of False Positives (true=neg, pred=pos):
- "First off let me say, If you haven't enjoyed a Van Damme movie since bloodsport, you probably will not like this movie. Most of these movies may not have the best plots or best actors but I enjoy thes"...

- 'Isaac Florentine has made some of the best western Martial Ar

In [14]:
import joblib

joblib.dump(tfidf, 'tfidf_vectorizer.joblib')
joblib.dump(clf,   'tfidf_lr_model.joblib')

['tfidf_lr_model.joblib']

Applying Pre-trained Model


In [15]:
from transformers import pipeline


# Instantiate a sentiment-analysis pipeline with a specific model

pipe = pipeline(
    task='sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english')



config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0


In [16]:
# Grab a small batch from test set for quick inference
texts = test_df['text'].tolist()[:8]

In [17]:

# Run the pipeline
preds = pipe(texts)

In [18]:

# Inspect the raw output
for txt, p in zip(texts, preds):
    print(f"Review excerpt: {txt[:60]!r}…")
    print(f" → label: {p['label']}, score: {p['score']:.4f}\n")

Review excerpt: 'I love sci-fi and am willing to put up with a lot. Sci-fi mo'…
 → label: NEGATIVE, score: 0.9996

Review excerpt: 'Worth the entertainment value of a rental, especially if you'…
 → label: NEGATIVE, score: 0.6171

Review excerpt: 'its a totally average film with a few semi-alright action se'…
 → label: NEGATIVE, score: 0.9997

Review excerpt: 'STAR RATING: ***** Saturday Night **** Friday Night *** Frid'…
 → label: NEGATIVE, score: 0.9958

Review excerpt: "First off let me say, If you haven't enjoyed a Van Damme mov"…
 → label: POSITIVE, score: 0.9963

Review excerpt: 'I had high hopes for this one until they changed the name to'…
 → label: NEGATIVE, score: 0.9967

Review excerpt: 'Isaac Florentine has made some of the best western Martial A'…
 → label: NEGATIVE, score: 0.9584

Review excerpt: 'It actually pains me to say it, but this movie was horrible '…
 → label: NEGATIVE, score: 0.9994



In [19]:
!pip install --quiet transformers datasets evaluate accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 22.7 MB/s eta 0:00:00


Fine tune Transformer


In [20]:
!pip install -q transformers datasets evaluate

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
import torch

In [21]:
ds = load_dataset("imdb")
ds = DatasetDict({
    "train": ds["train"].shuffle(seed=42).select(range(22500)),
    "validation": ds["train"].shuffle(seed=42).select(range(22500, 25000)),
    "test": ds["test"],
})

In [22]:
model_name = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

ds = ds.map(tokenize_batch, batched=True)
# Drop unneeded columns (raw text)
to_keep = ["input_ids", "attention_mask", "label"]
for split in ["train", "validation", "test"]:
    ds[split] = ds[split].remove_columns(
        [c for c in ds[split].column_names if c not in to_keep]
    )

# Rename "label" → "labels" for Trainer compatibility
ds = ds.rename_column("label", "labels")

# Format only three columns as torch tensors
ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/22500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [24]:
accuracy = evaluate.load("accuracy")
f1       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1":       f1.compute(predictions=preds, references=labels)["f1"],
    }


In [25]:
training_args = TrainingArguments(
    output_dir="imdb-distilbert-run1",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

    report_to="none",
    run_name=None,
)


In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.280400,0.224887,0.908800,0.911559
2,0.172900,0.285905,0.910800,0.911261
3,0.099300,0.327602,0.916800,0.919815


TrainOutput(global_step=4221, training_loss=0.1926606039785147, metrics={'train_runtime': 1539.4433, 'train_samples_per_second': 43.847, 'train_steps_per_second': 2.742, 'total_flos': 4470774704640000.0, 'train_loss': 0.1926606039785147, 'epoch': 3.0})

In [27]:
metrics = trainer.evaluate(ds["validation"])
print("Validation metrics:", metrics)

Validation metrics: {'eval_loss': 0.3276020586490631, 'eval_accuracy': 0.9168, 'eval_f1': 0.9198149575944488, 'eval_runtime': 16.8197, 'eval_samples_per_second': 148.635, 'eval_steps_per_second': 4.697, 'epoch': 3.0}


In [28]:
trainer.save_model("imdb-distilbert-finetuned")
tokenizer.save_pretrained("imdb-distilbert-finetuned")


('imdb-distilbert-finetuned/tokenizer_config.json',
 'imdb-distilbert-finetuned/special_tokens_map.json',
 'imdb-distilbert-finetuned/vocab.txt',
 'imdb-distilbert-finetuned/added_tokens.json',
 'imdb-distilbert-finetuned/tokenizer.json')

In [29]:
# Evaluate on the official test split
test_metrics = trainer.evaluate(ds["test"])
print("Test set metrics:", test_metrics)

Test set metrics: {'eval_loss': 0.3494538962841034, 'eval_accuracy': 0.91364, 'eval_f1': 0.9139600685450125, 'eval_runtime': 171.7369, 'eval_samples_per_second': 145.572, 'eval_steps_per_second': 4.553, 'epoch': 3.0}


In [31]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
 !apt-get install git-lfs

In [32]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load from local folder
model     = AutoModelForSequenceClassification.from_pretrained(
    "./imdb-distilbert-finetuned", local_files_only=True
)
tokenizer = AutoTokenizer.from_pretrained(
    "./imdb-distilbert-finetuned", local_files_only=True
)

# Push to your HF repo
model.push_to_hub(repo_id="rxnu/imdb-distilbert-finetuned")
tokenizer.push_to_hub(repo_id="rxnu/imdb-distilbert-finetuned")




Uploading...:   0%|          | 0.00/268M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/rxnu/imdb-distilbert-finetuned/commit/33f693c4c780a1970c96739239874e1eeed34612', commit_message='Upload tokenizer', commit_description='', oid='33f693c4c780a1970c96739239874e1eeed34612', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rxnu/imdb-distilbert-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='rxnu/imdb-distilbert-finetuned'), pr_revision=None, pr_num=None)

In [33]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [34]:
trainer.save_model("/content/drive/MyDrive/imdb-distilbert-finetuned")
tokenizer.save_pretrained("/content/drive/MyDrive/imdb-distilbert-finetuned")


('/content/drive/MyDrive/imdb-distilbert-finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/special_tokens_map.json',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/vocab.txt',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/added_tokens.json',
 '/content/drive/MyDrive/imdb-distilbert-finetuned/tokenizer.json')

In [35]:
!ls -l ./imdb-distilbert-finetuned


total 262512
-rw-r--r-- 1 root root       569 Jul  3 03:55 config.json
-rw-r--r-- 1 root root 267832560 Jul  3 03:55 model.safetensors
-rw-r--r-- 1 root root      5174 Jul  3 03:55 README.md
-rw-r--r-- 1 root root       695 Jul  3 03:55 special_tokens_map.json
-rw-r--r-- 1 root root      1418 Jul  3 03:55 tokenizer_config.json
-rw-r--r-- 1 root root    711661 Jul  3 03:55 tokenizer.json
-rw-r--r-- 1 root root      5304 Jul  3 03:51 training_args.bin
-rw-r--r-- 1 root root    231508 Jul  3 03:55 vocab.txt
